# 🔍 Discover your SAE's features — auto-label with LLM-judge

Given a trained SAE published to HuggingFace (from Tier 1/2/3), this notebook:

1. Loads the SAE + base model
2. Streams text, records per-feature activations
3. Picks the most interpretable features (mean-activation × fire-rate)
4. Asks an LLM (Claude or GPT) to label each feature from its top-activating context windows
5. Uploads `feature_catalog.json` back to the SAE repo

Runtime target: ≤ 20 min on a T4 for 30 features.

In [ ]:
!pip install -q transformers==4.57.1 accelerate==1.12.0 datasets==4.0.0 safetensors==0.4.5 huggingface_hub==1.5.0 anthropic openai tqdm

## Config

Edit `HF_SAE_REPO` to point at the SAE you want to label. Other defaults match a Gemma-2-2b SAE at layer 15 with d_sae=16384, K=64.

In [ ]:
HF_SAE_REPO       = 'YOUR_USER/your-sae'
HF_BASE_MODEL     = 'google/gemma-2-2b'
LAYER             = 15
D_MODEL           = 2304
D_SAE             = 16384
K                 = 64              # TopK SAE active features per token
N_FEATURES_TO_LABEL = 30
CONTEXT_TOKENS    = 20000           # tokens streamed from FineWeb-Edu
WINDOW            = 10              # tokens around each activation for context snippets

# Labeling backend — 'anthropic', 'openai', or 'auto'
JUDGE_BACKEND     = 'auto'
ANTHROPIC_MODEL   = 'claude-haiku-4-5'      # cheap: ~$0.05 for 30 features
OPENAI_MODEL      = 'gpt-4o-mini'
TOP_SNIPPETS      = 5               # snippets shown to the judge per feature

# Streaming / batching
SEQ_LEN           = 512
BATCH_SEQS        = 4
DATASET_NAME      = 'HuggingFaceFW/fineweb-edu'
DATASET_SUBSET    = 'sample-10BT'
DATASET_SPLIT     = 'train'

import os, json, math, time, random
random.seed(0)
print('Config loaded. SAE repo:', HF_SAE_REPO)

## Auth

Set `HF_TOKEN` (required — to read and upload) and at least one of `ANTHROPIC_API_KEY` / `OPENAI_API_KEY` (optional — to label). On Colab, use the 🔑 Secrets panel; outside Colab, read from env vars.

In [ ]:
def _get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        try:
            return userdata.get(name)
        except Exception:
            return None
    except Exception:
        return None

HF_TOKEN          = _get_secret('HF_TOKEN')
ANTHROPIC_API_KEY = _get_secret('ANTHROPIC_API_KEY')
OPENAI_API_KEY    = _get_secret('OPENAI_API_KEY')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(HF_TOKEN, add_to_git_credential=False)
    print('HF logged in.')
else:
    print('WARNING: no HF_TOKEN — upload at the end will fail.')

if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
    print('Anthropic key present.')
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print('OpenAI key present.')
if not (ANTHROPIC_API_KEY or OPENAI_API_KEY):
    print('No judge key — labeling will be skipped, catalog will still be saved with top snippets.')

In [ ]:
# --- Load SAE from HF ---
import torch
import torch.nn as nn
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, list_repo_files

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Try several common SAE filenames, in priority order.
_candidate_files = [
    'sae_final.safetensors',
    f'sae_L{LAYER}_latest.safetensors',
    f'sae_L{LAYER}.safetensors',
    'sae.safetensors',
]
repo_files = list_repo_files(HF_SAE_REPO)
sae_file = next((f for f in _candidate_files if f in repo_files), None)
if sae_file is None:
    # fallback: any .safetensors that looks like an SAE
    sae_file = next((f for f in repo_files if f.endswith('.safetensors') and 'sae' in f.lower()), None)
assert sae_file is not None, f'No SAE .safetensors found in {HF_SAE_REPO}. Files: {repo_files}'
print('Using SAE file:', sae_file)

sae_path = hf_hub_download(HF_SAE_REPO, sae_file)
sd = load_file(sae_path)
print('SAE state_dict keys:', list(sd.keys()))

class TopKSAE(nn.Module):
    """TopK SAE supporting both sae_lens (W_enc/W_dec/b_enc/b_dec) and encoder.weight fallback."""
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model = d_model
        self.d_sae = d_sae
        self.k = k
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    @torch.no_grad()
    def encode(self, x):
        # x: [..., d_model]  ->  [..., d_sae] sparse top-k
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = torch.topk(pre, self.k, dim=-1)
        vals = torch.relu(vals)
        out = torch.zeros_like(pre)
        out.scatter_(-1, idx, vals)
        return out

sae = TopKSAE(D_MODEL, D_SAE, K)

# Reconcile keys from multiple known formats.
def _assign(param_name, tensor):
    p = getattr(sae, param_name)
    if tensor.shape != p.shape:
        # handle transposed conventions
        if tensor.T.shape == p.shape:
            tensor = tensor.T
        else:
            raise ValueError(f'shape mismatch for {param_name}: got {tensor.shape}, expected {p.shape}')
    p.data.copy_(tensor.to(p.dtype))

aliases = {
    'W_enc': ['W_enc', 'encoder.weight', 'encoder.W', 'enc.weight'],
    'W_dec': ['W_dec', 'decoder.weight', 'decoder.W', 'dec.weight'],
    'b_enc': ['b_enc', 'encoder.bias',   'enc.bias'],
    'b_dec': ['b_dec', 'decoder.bias',   'dec.bias', 'pre_bias'],
}
for tgt, names in aliases.items():
    for n in names:
        if n in sd:
            _assign(tgt, sd[n])
            print(f'Loaded {tgt} <- {n} {tuple(sd[n].shape)}')
            break
    else:
        print(f'WARNING: no source found for {tgt} (left at zeros)')

sae = sae.to(device=device, dtype=torch.bfloat16).eval()
for p in sae.parameters():
    p.requires_grad_(False)
print('SAE ready on', device)

In [ ]:
# --- Load base model (bf16 + SDPA) and hook the chosen layer ---
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map={'': device},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Find the decoder layer list across common architectures.
_candidates = [
    getattr(getattr(model, 'model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None),
    getattr(getattr(model, 'language_model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'decoder', None), 'layers', None),
    getattr(getattr(model, 'transformer', None), 'h', None),
]
layers = next((c for c in _candidates if c is not None), None)
assert layers is not None, 'Could not locate decoder layers on this model.'
print(f'Hooking layer {LAYER} of {len(layers)} ({type(layers[LAYER]).__name__})')

_captured = {}
def _hook(module, inputs, output):
    h = output[0] if isinstance(output, tuple) else output
    _captured['h'] = h.detach()

hook_handle = layers[LAYER].register_forward_hook(_hook)
print('Hook registered.')

In [ ]:
# --- Stream text, compute SAE activations, accumulate per-feature top snippets ---
import heapq
from datasets import load_dataset
from tqdm.auto import tqdm

ds = load_dataset(DATASET_NAME, name=DATASET_SUBSET, split=DATASET_SPLIT, streaming=True)

# Running stats per feature:
# fire_count[f]         — tokens where feature fired (>0 after TopK)
# sum_activation[f]     — sum of activations when fired
# top_snippets[f]       — min-heap of (act, unique_tag, token_text_window, token_idx_in_doc)
fire_count     = torch.zeros(D_SAE, dtype=torch.float32)
sum_activation = torch.zeros(D_SAE, dtype=torch.float32)
top_snippets   = [ [] for _ in range(D_SAE) ]  # each: list kept as min-heap
HEAP_CAP       = TOP_SNIPPETS

total_tokens = 0
doc_counter = 0
batch_texts = []
pbar = tqdm(total=CONTEXT_TOKENS, desc='tokens streamed')

def _flush_batch(batch_texts):
    global total_tokens
    enc = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=SEQ_LEN)
    input_ids = enc['input_ids'].to(device)
    attn = enc['attention_mask'].to(device)
    with torch.no_grad():
        _ = model(input_ids=input_ids, attention_mask=attn, use_cache=False)
    h = _captured['h']                                  # [B, T, D_MODEL] bf16
    z = sae.encode(h.to(torch.bfloat16))                 # [B, T, D_SAE] bf16 sparse
    z = z.float().cpu()
    input_ids_cpu = input_ids.cpu()
    attn_cpu = attn.cpu().bool()
    B, T, _ = z.shape
    # Per-token accumulate
    for b in range(B):
        valid_t = int(attn_cpu[b].sum().item())
        if valid_t == 0:
            continue
        zb = z[b, :valid_t]                              # [valid_t, D_SAE]
        ids_b = input_ids_cpu[b, :valid_t].tolist()
        # Vectorized stats across all tokens in this sequence
        fired_mask = (zb > 0)
        fire_count += fired_mask.sum(dim=0).float()
        sum_activation += zb.sum(dim=0)
        total_tokens += valid_t
        pbar.update(valid_t)
        # Top snippets: only look at tokens where something fired and then pick topk features per token.
        # Much cheaper than scanning all D_SAE features.
        acts_vals, acts_idx = zb.topk(K, dim=-1)          # [valid_t, K]
        for t in range(valid_t):
            lo = max(0, t - WINDOW)
            hi = min(valid_t, t + WINDOW + 1)
            try:
                snippet = tokenizer.decode(ids_b[lo:hi], skip_special_tokens=True)
            except Exception:
                snippet = ''
            snippet = snippet.replace('\n', ' ')[:400]
            tag = (doc_counter, b, t)
            vk = acts_vals[t].tolist()
            ik = acts_idx[t].tolist()
            for v, f in zip(vk, ik):
                if v <= 0:
                    continue
                heap = top_snippets[f]
                if len(heap) < HEAP_CAP:
                    heapq.heappush(heap, (v, tag, snippet))
                elif v > heap[0][0]:
                    heapq.heapreplace(heap, (v, tag, snippet))

for row in ds:
    txt = row.get('text') or row.get('content') or ''
    if not txt:
        continue
    batch_texts.append(txt)
    doc_counter += 1
    if len(batch_texts) >= BATCH_SEQS:
        _flush_batch(batch_texts)
        batch_texts = []
        if total_tokens >= CONTEXT_TOKENS:
            break
if batch_texts and total_tokens < CONTEXT_TOKENS:
    _flush_batch(batch_texts)
pbar.close()
hook_handle.remove()
print(f'Done. total_tokens={total_tokens}, docs={doc_counter}')

In [ ]:
# --- Rank features by mean-activation-when-fired × fire-rate ---
eps = 1e-9
fire_rate = fire_count / max(total_tokens, 1)
mean_when_fired = torch.where(fire_count > 0, sum_activation / (fire_count + eps), torch.zeros_like(fire_count))
# Score balances "bland but frequent" vs "rare but loud".
# Log-fire-rate keeps dead features out but doesn't reward pure frequency.
log_rate = torch.log1p(fire_count)
score = mean_when_fired * log_rate
# Require at least TOP_SNIPPETS firings to be labeled (otherwise the judge has nothing to see)
viable = (fire_count >= TOP_SNIPPETS).nonzero(as_tuple=False).flatten().tolist()
viable_set = set(viable)
sorted_feats = sorted(range(D_SAE), key=lambda f: -score[f].item())
chosen = [f for f in sorted_feats if f in viable_set][:N_FEATURES_TO_LABEL]
print(f'{len(viable)} viable features (>= {TOP_SNIPPETS} firings). Labeling top {len(chosen)}.')
print('\nrank  id     fire_rate    mean_act   score')
for r, f in enumerate(chosen):
    print(f'{r:4d}  {f:6d}  {fire_rate[f].item():.6f}  {mean_when_fired[f].item():8.4f}  {score[f].item():8.4f}')

## LLM-judge labeling

For each chosen feature we send its top `TOP_SNIPPETS` context windows to an LLM and ask for `{name, desc, confidence}`.

**Cost estimate** (30 features × ~500 input + ~80 output tokens each):

| Model                  | Approx cost |
|------------------------|-------------|
| Claude Haiku 4.5       | ~$0.05      |
| Claude Sonnet 4        | ~$0.50      |
| GPT-4o-mini (OpenAI)   | ~$0.02      |

If no API key is set the notebook skips labeling and still saves the catalog with raw top-activating snippets.

In [ ]:
# --- Build the judge backend ---
def _pick_backend():
    if JUDGE_BACKEND == 'anthropic' and ANTHROPIC_API_KEY:
        return 'anthropic'
    if JUDGE_BACKEND == 'openai' and OPENAI_API_KEY:
        return 'openai'
    if JUDGE_BACKEND == 'auto':
        if ANTHROPIC_API_KEY:
            return 'anthropic'
        if OPENAI_API_KEY:
            return 'openai'
    return None

backend = _pick_backend()
print('Judge backend:', backend or 'NONE (labeling skipped)')

def _build_prompt(snippets):
    lines = []
    for i, s in enumerate(snippets, 1):
        lines.append(f'[{i}] {s}')
    joined = '\n'.join(lines)
    return (
        'You are inspecting a feature from a sparse autoencoder over a language model.\n'
        'This feature activates most strongly on the following text snippets (the activation '
        'is centered roughly in the middle of each snippet).\n\n'
        f'{joined}\n\n'
        'Give a one-sentence semantic description of what this feature seems to detect.\n'
        'Respond ONLY with minified JSON of the form:\n'
        '{"name": "snake_case_name", "desc": "one sentence", "confidence": 0.0-1.0}'
    )

def _parse_json(text):
    import re
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return None
    if not isinstance(obj, dict):
        return None
    name = str(obj.get('name', 'unnamed'))[:60]
    desc = str(obj.get('desc', ''))[:300]
    try:
        conf = float(obj.get('confidence', 0.0))
    except Exception:
        conf = 0.0
    conf = max(0.0, min(1.0, conf))
    return {'name': name, 'desc': desc, 'confidence': conf}

def _label_anthropic(prompt):
    import anthropic
    client = anthropic.Anthropic()
    r = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return r.content[0].text

def _label_openai(prompt):
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(
        model=OPENAI_MODEL,
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return r.choices[0].message.content

def label_feature(snippets):
    if backend is None:
        return {'name': 'unlabeled', 'desc': 'skipped — configure ANTHROPIC_API_KEY or OPENAI_API_KEY and rerun', 'confidence': 0.0}
    prompt = _build_prompt(snippets)
    try:
        raw = _label_anthropic(prompt) if backend == 'anthropic' else _label_openai(prompt)
    except Exception as e:
        return {'name': 'error', 'desc': f'judge call failed: {e}'[:300], 'confidence': 0.0}
    parsed = _parse_json(raw)
    return parsed or {'name': 'parse_error', 'desc': raw[:300], 'confidence': 0.0}

# --- Build catalog entries ---
catalog = []
for rank, f in enumerate(tqdm(chosen, desc='labeling')):
    heap = sorted(top_snippets[f], key=lambda x: -x[0])
    snippets = [s for (_, _, s) in heap][:TOP_SNIPPETS]
    labeled = label_feature(snippets) if snippets else {'name': 'empty', 'desc': 'no firings', 'confidence': 0.0}
    catalog.append({
        'id': int(f),
        'rank': int(rank),
        'name': labeled['name'],
        'desc': labeled['desc'],
        'confidence': labeled['confidence'],
        'fire_rate': float(fire_rate[f].item()),
        'mean_when_fired': float(mean_when_fired[f].item()),
        'score': float(score[f].item()),
        'top_activating_examples': [
            {'activation': float(v), 'text': s} for (v, _, s) in heap[:TOP_SNIPPETS]
        ],
    })

print(f'Labeled {len(catalog)} features.')
for e in catalog[:5]:
    print(f"  [{e['id']}] {e['name']} (conf={e['confidence']:.2f}) — {e['desc']}")

In [ ]:
# --- Save + upload feature_catalog.json to the SAE repo ---
from huggingface_hub import HfApi

out_path = 'feature_catalog.json'
payload = {
    'sae_repo': HF_SAE_REPO,
    'base_model': HF_BASE_MODEL,
    'layer': LAYER,
    'd_model': D_MODEL,
    'd_sae': D_SAE,
    'k': K,
    'context_tokens_analyzed': int(total_tokens),
    'judge_backend': backend or 'none',
    'judge_model': (ANTHROPIC_MODEL if backend == 'anthropic' else (OPENAI_MODEL if backend == 'openai' else None)),
    'features': catalog,
}
with open(out_path, 'w') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print('Wrote', out_path, f'({os.path.getsize(out_path)} bytes)')

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo='feature_catalog.json',
        repo_id=HF_SAE_REPO,
        repo_type='model',
        commit_message=f'Add feature_catalog.json ({len(catalog)} labeled features)',
    )
    print(f'Uploaded to https://huggingface.co/{HF_SAE_REPO}/blob/main/feature_catalog.json')
else:
    print('Skipping upload — no HF_TOKEN. Download feature_catalog.json manually.')